# Arithmetic

EasyScience numbers know their **unit**, their **uncertainty** and, for a `Parameter`, the **bounds** a fit
may explore. The operators `+`, `-`, `*`, `/` and `**`, negation and `abs` work on them and on plain
numbers, and every result is a **new** object whose value, unit, variance and bounds are worked out for you.

This notebook shows how that works and how it is expected to behave. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

For independent inputs the variance of a result is propagated to first order,

$$
\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 .
$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The examples below print numbers with f-strings, so that every value, uncertainty and unit is visible. The solutions of the exercises use a small function, `show`, that prints a number as one line: value, error and unit and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## Numbers with an uncertainty

A `Parameter` has a value, a unit, an uncertainty and bounds. Here is a length of 2 m with an uncertainty of
0.2 m, which a fit may vary between 0 and 10 m. The uncertainty is given as a variance, so it is written as
the square of the error, $0.2^2 = 0.04\ \text{m}^2$:

In [ ]:
length = Parameter(display_name='length', value=2.0, unit='m', variance=0.2**2, min=0.0, max=10.0)
length

The uncertainty is stored as a **variance**, in the square of the unit, and `error` is its square root, in the unit itself:

In [ ]:
print(f'value:    {length.value} {length.unit}')
print(f'error:    {length.error:.2f} {length.unit}')
print(f'variance: {length.variance:.2f} {length.unit}²')
print(f'bounds:   [{length.min}, {length.max}] {length.unit}')

A `DescriptorNumber` is the same without the fitting bounds. It is the right choice for a number that is
known but is never fitted, such as a measured constant:

In [ ]:
scale = DescriptorNumber(display_name='scale', value=3.0, variance=0.1**2)
print(f'scale: {scale.value:.2f} ± {scale.error:.2f}')

### Error and uncertainty

In everyday language the two words are used for the same thing, and so does EasyScience: the property that
holds the square root of the variance is called `error`. Strictly speaking they are not the same.

- The **error** of a measurement is the difference between the measured value and the true value. The true
  value is usually unknown, so the error of a real measurement can seldom be stated.
- The **uncertainty** is a quantitative estimate of the range around the measured value in which the true
  value is expected to lie. It describes the precision, or the doubt, of the measurement.

What EasyScience stores is an uncertainty, as a variance, and `error` is the standard uncertainty: one standard deviation. The standard deviation is a complete summary of the uncertainty only when the distribution of the values is symmetric and close to Gaussian. Then $\pm 1\sigma$ covers about 68 % of the possible values, and "error" and "uncertainty" can be used loosely. This is **not always the case**: for a skewed or heavy-tailed distribution, for a systematic effect that no repeated measurement reveals, or for a result that has gone through a strongly non-linear function, the same $\pm$ number can be misleading. What the word means depends on the context.

## Addition and subtraction

The operators work between `Parameter`s, `DescriptorNumber`s and plain numbers. The result is a `Parameter`
(with bounds) if any operand is a `Parameter`, and a `DescriptorNumber` otherwise.

Only numbers with the same physical dimension can be added. If the units differ, the second operand is
converted to the unit of the first, so the result is in metres here:

In [ ]:
width = Parameter(
    display_name='width', value=50.0, unit='cm', variance=2.0**2, min=10.0, max=100.0
)
total = length + width

print(f'length: {length.value:.2f} ± {length.error:.2f} {length.unit}')
print(f'width:  {width.value:.2f} ± {width.error:.2f} {width.unit}')
print(f'total:  {total.value:.4f} ± {total.error:.4f} {total.unit}')
print(f'variance of the total: {total.variance:.4f} {total.unit}²')
print(f'bounds of the total:   [{total.min:.2f}, {total.max:.2f}] {total.unit}')

The width is converted to metres first: $0.50 \pm 0.02$ m. The variances then add, each one in square metres:

$$
(0.2\ \text{m})^2 + (0.02\ \text{m})^2 = 0.04\ \text{m}^2 + 0.0004\ \text{m}^2 = 0.0404\ \text{m}^2 ,
$$

so the error of the total is $\sqrt{0.0404\ \text{m}^2} = 0.2010$ m. The bounds add as well, in metres: $[0 + 0.1,\ 10 + 1.0]$ m.
Subtraction combines the bounds the other way round: the lowest possible difference is the minimum of the
first minus the maximum of the second, here $0 - 1.0 = -1.0$ m. The variances still add:

In [ ]:
difference = length - width

print(f'difference: {difference.value:.4f} ± {difference.error:.4f} {difference.unit}')
print(f'variance of the difference: {difference.variance:.4f} {difference.unit}²')
print(
    f'bounds of the difference:   [{difference.min:.2f}, {difference.max:.2f}] {difference.unit}'
)

A plain number can only be added to a dimensionless value. Adding one to a length is an error, while a
dimensionless `DescriptorNumber` accepts it:

In [ ]:
try:
    length + 1.0
except Exception as error:
    print(f'{type(error).__name__}: {error}')

shifted = scale + 1.0
print(f'scale + 1: {shifted.value:.2f} ± {shifted.error:.2f}')

### Exercise 1: Total length

A rod is built from two sections that were measured separately: 1.20 ± 0.02 m and 0.80 ± 0.03 m. What is the total length, with its uncertainty?

In [ ]:
section_1 = Parameter(display_name='section_1', value=1.20, variance=0.02**2, unit='m')
section_2 = Parameter(display_name='section_2', value=0.80, variance=0.03**2, unit='m')

<details class="question">
<summary>Hint 1</summary>

Two lengths in the same unit can simply be added with `+`.

</details>

<details class="question">
<summary>Hint 2</summary>

The variances add: $\sigma^2 = 0.02^2 + 0.03^2$, and `error` is the square root of the variance.

</details>

In [ ]:
%%solution
total = section_1 + section_2
print(show(total))

### Exercise 2: Mixed units

A tube is 1.20 ± 0.02 m long and a gap of 12.5 ± 0.5 cm has to be cut out of it. How long is the rest, in metres?

In [ ]:
tube = Parameter(display_name='tube', value=1.20, variance=0.02**2, unit='m')
gap = Parameter(display_name='gap', value=12.5, variance=0.5**2, unit='cm')

<details class="question">
<summary>Hint 1</summary>

Subtract the gap from the tube; the units do not have to match.

</details>

<details class="question">
<summary>Hint 2</summary>

The second operand is converted to the unit of the first, so the result is in the unit of `tube`.

</details>

In [ ]:
%%solution
remaining = tube - gap
print(show(remaining))

## Multiplication and division

Units multiply and divide. The area of a rectangle, with the sides in the same unit:

In [ ]:
height = Parameter(display_name='height', value=1.5, unit='m', variance=0.1**2, min=1.0, max=3.0)
area = length * height

print(f'area: {area.value:.4f} ± {area.error:.4f} {area.unit}')
print(f'bounds of the area: [{area.min:.2f}, {area.max:.2f}] {area.unit}')

The bounds of a product are the lowest and highest of the four products of the bounds. A speed is a length
over a time:

In [ ]:
time = Parameter(display_name='time', value=4.0, unit='s', variance=0.25**2, min=2.0, max=8.0)
speed = length / time

print(f'speed: {speed.value:.4f} ± {speed.error:.4f} {speed.unit}')
print(f'bounds of the speed: [{speed.min:.2f}, {speed.max:.2f}] {speed.unit}')

Multiplying a `DescriptorNumber` by a plain number gives a `DescriptorNumber`, so there are no bounds:

In [ ]:
doubled = scale * 2
print(f'{type(doubled).__name__}: {doubled.value:.2f} ± {doubled.error:.2f}')

### Exercise 3: Speed

A cart covers 3.0 ± 0.1 m in 1.5 ± 0.05 s. What is its speed, with its uncertainty and unit?

In [ ]:
distance = Parameter(display_name='distance', value=3.0, variance=0.1**2, unit='m')
time = Parameter(display_name='time', value=1.5, variance=0.05**2, unit='s')

<details class="question">
<summary>Hint 1</summary>

Speed is distance divided by time.

</details>

<details class="question">
<summary>Hint 2</summary>

Units divide as well: look at the unit of the result.

</details>

In [ ]:
%%solution
speed = distance / time
print(show(speed))

## Powers, negation and absolute value

The exponent of a power is a plain number. Negation flips the bounds, and `abs` folds them:

In [ ]:
squared = length**2
negated = -length
absolute = abs(-length)

print(
    f'length**2: {squared.value:.2f} ± {squared.error:.2f} {squared.unit}, bounds [{squared.min:.1f}, {squared.max:.1f}]'
)
print(
    f'-length:   {negated.value:.2f} ± {negated.error:.2f} {negated.unit}, bounds [{negated.min:.1f}, {negated.max:.1f}]'
)
print(
    f'abs(-length): {absolute.value:.2f} ± {absolute.error:.2f} {absolute.unit}, bounds [{absolute.min:.1f}, {absolute.max:.1f}]'
)

### Exercise 4: Area of a beam spot

The radius of a circular beam spot is 2.0 ± 0.1 cm. What is its area?

In [ ]:
radius = Parameter(display_name='radius', value=2.0, variance=0.1**2, unit='cm')

<details class="question">
<summary>Hint 1</summary>

The area is $\pi r^2$. `np.pi` is a plain number, and `**` takes a plain-number exponent.

</details>

<details class="question">
<summary>Hint 2</summary>

A plain number can multiply a `Parameter`; the result is again a `Parameter`.

</details>

In [ ]:
%%solution The lower bound 0 appears because a square can never be negative.
area = np.pi * radius**2
print(show(area))

## Uncertainty in a chart

The uncertainty of a result depends on the inputs. A speed $v = d/t$ of an object that covered
$3.0 \pm 0.1$ m is much less certain for short times than for long ones, even though the time has the same
uncertainty of 0.05 s each time. The chart is interactive: hover over the curve to read the values. It loads
Plotly from the internet, so it needs a connection. The charts are shown with a small helper,
`show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the same.

In [ ]:
times = np.linspace(0.5, 5.0, 46)
distance = Parameter(display_name='distance', value=3.0, unit='m', variance=0.1**2)

values, errors = [], []
for value in times:
    time = Parameter(display_name='time', value=value, unit='s', variance=0.05**2)
    speed = distance / time
    values.append(speed.value)
    errors.append(speed.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=times,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=values,
        mode='lines',
        name='speed',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='time = %{x:.2f} s<br>speed = %{y:.2f} ± %{customdata:.2f} m/s<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='time / s',
    yaxis_title='speed / (m/s)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.6, y=0.95),
)
show_figure(fig)

Variances add, so the error of a **sum** of $n$ equal measurements grows like $\sigma\sqrt{n}$, while the
error of their **mean** shrinks like $\sigma/\sqrt{n}$. Here each measurement is $1.0 \pm 0.1$:

In [ ]:
counts = np.arange(1, 26)

sum_errors, mean_errors = [], []
for n in counts:
    total = Parameter(display_name='measurement', value=1.0, variance=0.1**2)
    for _ in range(int(n) - 1):
        total = total + Parameter(display_name='measurement', value=1.0, variance=0.1**2)
    sum_errors.append(total.error)
    mean_errors.append((total / int(n)).error)

fig = go.Figure()
fig.add_trace(go.Scatter(x=counts, y=sum_errors, mode='lines+markers', name='error of the sum'))
fig.add_trace(go.Scatter(x=counts, y=mean_errors, mode='lines+markers', name='error of the mean'))
fig.update_layout(
    xaxis_title='number of measurements n',
    yaxis_title='error',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig, height=380)

## Good to know

A result remembers which numbers it was calculated from. The same quantity reached by different routes is one
quantity: $x - x$ is exactly $0 \pm 0$, and so is $(2x)/x - 2$, because the $x$ cancels. Bounds are different.
They are combined as if the operands were independent, so $x - x$ of a `Parameter` that may vary between 0 and
10 has bounds $[-10, 10]$:

In [ ]:
x = Parameter(display_name='x', value=10.0, variance=0.1**2, min=0.0, max=10.0)
difference = x - x
ratio = (2 * x) / x

print(
    f'x - x:     {difference.value:.4f} ± {difference.error:.4f}, bounds [{difference.min:.1f}, {difference.max:.1f}]'
)
print(f'(2x) / x:  {ratio.value:.4f} ± {ratio.error:.4f}')

Quantities that are correlated, such as the slope and the intercept of one fit, share a covariance. After a
fit the arithmetic uses it. Numbers that come without a shared covariance are treated as independent; the
notebook on correlation shows how to state one.